# Load Testing LLM APIs

**Phase 07** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-07/10-load-testing-llm-apis.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your team ships a new AI feature. Traffic ramps up post-launch. At 50 concurrent users, response times jump from 2s to 18s and some requests start returning 429 errors. The on-call engineer has never seen this failure mode.

Standard HTTP load testers like k6 or wrk will tell you requests/sec and error rate. But LLM APIs have three properties that make standard load testing misleading:

First, response times are variable by 10-100x depending on output length. A 50-token response takes 1s; a 2,000-token response takes 20s. Treating these as the same "request" in a p99 measurement hides the actual latency distribution.

Second, streaming responses mean time-to-first-token (TTFT) is a distinct ...

## The Concept

### Concurrent Request Timing

```
Time (seconds) -->

req-01  |====QUEUE===|=TTFT=|=========STREAM=========|  done at t=12
req-02  |====QUEUE===|=TTFT=|======STREAM======|        done at t=9
req-03  |====QUEUE===|==TTFT==|============STREAM============|  done at t=16
req-04            |=QUEUE=|=TTFT=|====STREAM====|            done at t=11
req-05            |=QUEUE=|==TTFT==|=========STREAM=========| done at t=15

         ^ concurrency starts here
                     ^ first token for each request (TTFT = queue + model startup)
                                   ^ streaming output (cost of ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
LLM Load Tester - Phase 07, Lesson 10
Measures TTFT, total latency, error rate, and estimated cost for LLM API calls.

Usage:
    # Mock mode (no API key needed)
    python main.py --mock --concurrency 10 --requests 50

    # Real API mode (requires ANTHROPIC_API_KEY)
    ANTHROPIC_API_KEY=sk-ant-... python main.py --concurrency 5 --requests 20
"""

from __future__ import annotations

import argparse
import asyncio
import dataclasses
import os
import random
import statistics
import time
from typing import Optional


@dataclasses.dataclass
class LoadTestConfig:
    concurrency: int = 10
    total_requests: int = 50
    prompt: str = "Explain the tradeoffs between synchronous and asynchronous Python in 100 words."
    max_tokens: int = 150
    model: str = "claude-3-5-haiku-20241022"
    use_mock: bool = True
    timeout_seconds: float = 30.0


@dataclasses.dataclass
class RequestResult:
    request_id: int
    ttft_ms: float          # Time to first token (ms)
    total_latency_ms: float # Time to last token (ms)
    input_tokens: int
    output_tokens: int
    status: str             # "ok", "timeout", "rate_limited", "error"
    error_message: Optional[str] = None


@dataclasses.dataclass
class LoadTestReport:
    config: LoadTestConfig
    results: list[RequestResult]
    wall_time_seconds: float

    def _percentile(self, values: list[float], p: float) -> float:
        if not values:
            return 0.0
        sorted_vals = sorted(values)
        idx = int(len(sorted_vals) * p / 100)
        idx = min(idx, len(sorted_vals) - 1)
        return sorted_vals[idx]

    def print_summary(self):
        total = len(self.results)
        ok = [r for r in self.results if r.status == "ok"]
        errors = [r for r in self.results if r.status != "ok"]
        timeouts = [r for r in self.results if r.status == "timeout"]
        rate_limits = [r for r in self.results if r.status == "rate_limited"]

        ttft_vals = [r.ttft_ms for r in ok]
        total_lat_vals = [r.total_latency_ms for r in ok]

        throughput = total / self.wall_time_seconds if self.wall_time_seconds > 0 else 0

        # Cost estimate (haiku pricing: $1/M input, $5/M output tokens)
        input_tokens = sum(r.input_tokens for r in ok)
        output_tokens = sum(r.output_tokens for r in ok)
        est_cost = (input_tokens * 0.000001) + (output_tokens * 0.000005)

        print("\nLLM Load Test Report")
        print("====================")
        print(f"Config: {total} requests, concurrency={self.config.concurrency}, mock={self.config.use_mock}")
        print(f"Duration: {self.wall_time_seconds:.1f}s  |  Throughput: {throughput:.1f} req/s")

        print("\nLatency (TTFT)")
        if ttft_vals:
            print(f"  p50:  {self._percentile(ttft_vals, 50):6.0f}ms")
            print(f"  p95:  {self._percentile(ttft_vals, 95):6.0f}ms")
            print(f"  p99:  {self._percentile(ttft_vals, 99):6.0f}ms")
        else:
            print("  No successful requests")

        print("\nLatency (Total)")
        if total_lat_vals:
            print(f"  p50:  {self._percentile(total_lat_vals, 50):6.0f}ms")
            print(f"  p95:  {self._percentile(total_lat_vals, 95):6.0f}ms")
            print(f"  p99:  {self._percentile(total_lat_vals, 99):6.0f}ms")
        else:
            print("  No successful requests")

        error_pct = len(errors) / total * 100 if total > 0 else 0
        print(f"\nErrors")
        print(f"  Total:       {len(errors):3d}  ({error_pct:.1f}%)")
        print(f"  Timeouts:    {len(timeouts):3d}")
        print(f"  Rate limits: {len(rate_limits):3d}")

        print(f"\nCost estimate ({'mock - est based on config' if self.config.use_mock else 'real API'})")
        print(f"  Input tokens:  ~{input_tokens:,}")
        print(f"  Output tokens: ~{output_tokens:,}")
        print(f"  Est. cost:     ${est_cost:.4f} (haiku pricing)")


class MockLLMClient:
    """Simulates LLM API latency without making real calls."""

    async def stream_request(self, prompt: str, max_tokens: int) -> RequestResult:
        req_id = random.randint(1000, 9999)
        start = time.perf_counter()

        # Simulate TTFT: 30-120ms
        await asyncio.sleep(random.uniform(0.03, 0.12))
        ttft_ms = (time.perf_counter() - start) * 1000

        # Simulate streaming: proportional to max_tokens
        output_tokens = random.randint(max_tokens // 2, max_tokens)
        stream_duration = output_tokens * 0.002  # ~2ms per output token
        await asyncio.sleep(stream_duration)
        total_ms = (time.perf_counter() - start) * 1000

        # Simulate occasional errors (2% rate limit, 1% timeout)
        rand = random.random()
        if rand < 0.01:
            return RequestResult(
                request_id=req_id, ttft_ms=0, total_latency_ms=total_ms,
                input_tokens=0, output_tokens=0, status="timeout",
                error_message="Simulated timeout"
            )
        if 0.01 <= rand < 0.03:
            return RequestResult(
                request_id=req_id, ttft_ms=0, total_latency_ms=total_ms,
                input_tokens=0, output_tokens=0, status="rate_limited",
                error_message="Simulated 429"
            )

        input_tokens = len(prompt) // 4
        return RequestResult(
            request_id=req_id,
            ttft_ms=ttft_ms,
            total_latency_ms=total_ms,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            status="ok",
        )


class AnthropicStreamClient:
    """Real Anthropic streaming client that measures TTFT."""

    def __init__(self, api_key: str):
        import anthropic
        self.client = anthropic.AsyncAnthropic(api_key=api_key)

    async def stream_request(
        self, model: str, prompt: str, max_tokens: int, request_id: int
    ) -> RequestResult:
        start = time.perf_counter()
        ttft_ms: Optional[float] = None
        output_tokens = 0

        try:
            async with self.client.messages.stream(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
            ) as stream:
                async for chunk in stream:
                    if ttft_ms is None:
                        ttft_ms = (time.perf_counter() - start) * 1000
                    # Count approximate output tokens from text chunks
                    if hasattr(chunk, "delta") and hasattr(chunk.delta, "text"):
                        output_tokens += len(chunk.delta.text) // 4

                final_message = await stream.get_final_message()
                total_ms = (time.perf_counter() - start) * 1000
                input_tokens = final_message.usage.input_tokens
                output_tokens = final_message.usage.output_tokens

            return RequestResult(
                request_id=request_id,
                ttft_ms=ttft_ms or total_ms,
                total_latency_ms=total_ms,
                input_tokens=input_tokens,
                output_tokens=output_tokens,
                status="ok",
            )
        except Exception as e:
            total_ms = (time.perf_counter() - start) * 1000
            error_str = str(e)
            status = "rate_limited" if "429" in error_str else "error"
            return RequestResult(
                request_id=request_id,
                ttft_ms=0, total_latency_ms=total_ms,
                input_tokens=0, output_tokens=0,
                status=status, error_message=error_str[:100],
            )


class LLMLoadTester:
    def __init__(self, config: LoadTestConfig):
        self.config = config

    async def run(self) -> LoadTestReport:
        semaphore = asyncio.Semaphore(self.config.concurrency)
        results: list[RequestResult] = []
        mock_client = MockLLMClient()

        api_key = os.environ.get("ANTHROPIC_API_KEY", "")
        real_client = AnthropicStreamClient(api_key) if (not self.config.use_mock and api_key) else None

        async def run_single(req_id: int) -> RequestResult:
            async with semaphore:
                if self.config.use_mock or real_client is None:
                    return await mock_client.stream_request(
                        self.config.prompt, self.config.max_tokens
                    )
                else:
                    return await real_client.stream_request(
                        self.config.model, self.config.prompt,
                        self.config.max_tokens, req_id
                    )

        print(f"Starting load test: {self.config.total_requests} requests, "
              f"concurrency={self.config.concurrency}, mock={self.config.use_mock}")

        start = time.perf_counter()
        tasks = [run_single(i) for i in range(self.config.total_requests)]
        results = await asyncio.gather(*tasks, return_exceptions=False)
        wall_time = time.perf_counter() - start

        return LoadTestReport(
            config=self.config,
            results=list(results),
            wall_time_seconds=wall_time,
        )


async def main_async(args):
    config = LoadTestConfig(
        concurrency=args.concurrency,
        total_requests=args.requests,
        use_mock=args.mock,
        max_tokens=150,
    )
    tester = LLMLoadTester(config)
    report = await tester.run()
    report.print_summary()


def main():
    parser = argparse.ArgumentParser(description="LLM API load tester")
    parser.add_argument("--mock", action="store_true", default=True,
                        help="Use mock client (no API calls)")
    parser.add_argument("--no-mock", dest="mock", action="store_false",
                        help="Use real Anthropic API (requires ANTHROPIC_API_KEY)")
    parser.add_argument("--concurrency", type=int, default=10,
                        help="Number of concurrent requests")
    parser.add_argument("--requests", type=int, default=50,
                        help="Total number of requests to send")
    args = parser.parse_args()
    asyncio.run(main_async(args))

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is wrong with this load test approach?**

- A. wrk is the wrong tool - you should use k6 instead.
- B. wrk measures HTTP throughput but not TTFT or streaming behavior. A 'successful' response to wrk might be a rate limit that returns HTTP 200 with an error body. You also risk spending $50+ to validate a broken test setup.
- C. 100 threads is too many for wrk. Start at 10.
- D. The Anthropic API cannot be load tested directly.

**2. What does this data tell you about your safe operating limit and what should you do?**

- A. The safe limit is 30 concurrency. Rate limits start at 25.
- B. The safe limit is around 25 concurrency. The jump from 0.8% to 8% error rate between 25 and 30 indicates you are near the rate limit boundary. Set your auto-scaling trigger at 70% of 25 (about 17-18 concurrent users) and add retry logic with backoff for the remaining 2%.
- C. Rate limits are expected at any concurrency. Configure retries and run at 50 concurrency.
- D. Contact Anthropic to raise your rate limits before proceeding.

**3. What does the data actually show and what should you investigate?**

- A. The colleague is right. p50=140ms is excellent.
- B. p50=140ms is correct, but the 4500ms outlier is a critical signal. One request out of 10 took 30x longer than the median. This is either a rate limit retry, a cold start, or a long-tail output length issue. At scale, this 10% outlier becomes your p90 - investigate before launch.
- C. The data is invalid because the variance is too high for LLM APIs.
- D. The 4500ms request is a network timeout and should be excluded from the analysis.

**4. Is this result expected, and what does it tell you?**

- A. The real API results indicate a bug in the harness. The numbers should be similar to mock.
- B. This is expected and correct. Mock validates that the harness logic is right (0% error, correct metrics collection). The real API numbers reflect actual network latency (~1800ms TTFT), real model inference time, and real rate limits. The 3% error rate at this concurrency level means you have found the rate limit boundary.
- C. The mock is misconfigured. Real APIs should be faster than mock.
- D. The real API test should be aborted - 3% error rate means the API is down.

**5. What is the correct approach?**

- A. Run load tests in production during low-traffic hours to minimize user impact.
- B. Never load test in production. Build a realistic staging environment instead.
- C. Run load tests in staging with a small real API integration: staging application stack, real Anthropic API, conservative request counts (20-50 req). This gives realistic model latency without production risk and controlled cost.
- D. Load test the mock staging environment and multiply latency by a realistic factor.

**6. What is the most likely root cause of the p95 violation and what is the correct fix?**

- A. The API is slow. Request a latency SLA from Anthropic.
- B. The p95 violation is driven by the high-output-token tail. Requests generating 1800 output tokens at ~15ms/token take 27 seconds of streaming alone. Set max_tokens=500 or add explicit instructions to limit response length, then re-run the load test.
- C. Increase concurrency to spread load more evenly.
- D. Use a faster model (Sonnet instead of Haiku) to reduce streaming time.

**7. Why is a high spawn-rate problematic for LLM load tests specifically?**

- A. Locust crashes when spawn-rate matches total users.
- B. Starting all users simultaneously creates a thundering herd that hits the rate limit immediately, before you can observe the gradual degradation that mirrors real traffic growth. You learn 'everything breaks at 100 users' but not 'performance degrades starting at 40 users'. A spawn-rate of 2-5 users/second gives you the degradation curve.
- C. High spawn-rate wastes money because requests overlap.
- D. The Anthropic API enforces a maximum of 10 new connections per second.

_Answers are in `checks.json` in the lesson directory._